# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb)

**Lane 2 — Refresh / Content Opportunity Scoring.** This week turns the audit lens on my own earlier work: two
findings from the FlyRank research paper (`docs/flyrank-seo-research-march-2026.pdf`) are read for methodology, my
Week-5 model is re-run under an honest grouped-by-client split with a **before/after** table (random split vs grouped
split), the same leakage hunt from Week 3 is re-run on my **final** feature set, and the boldest sentence from my Week-1
framing is rewritten in safe claim language.

Skill: `hunting-leakage-and-validating` + `flyrank/flyrank-data` (loaded from `skills/README.md`).


## 1. Two paper findings + my methodology questions

*For each finding: where does the label come from, and does the validation design carry the claim? Constructive tone —
the way I would want my own work reviewed.*

### Finding A — "The Anatomy of Growing Content" (paper Finding #1, CONFIRMED)

The paper's simplest comparison says:

> **Paper:** "Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days)... The sample
> sizes here are large (74,187 rising vs 45,272 falling), so the word-count and age gap is directionally robust even
> though this remains an observational comparison." — then "Expected: Improves the odds that an already visible page can
> keep growing rather than plateauing."

**Where does the label come from?** "Growing" vs "declining" is the paper's own `Trend Direction` flag —
> **Paper:** "Calculated from 30d-vs-prev-30d impression change. Up: >10% growth. Down: >10% decline."

That is the *same label family* as my `is_declining_label` (both come from a trend on impressions in one 90-day
window). Word count and age are then compared *after* grouping by that label — a two-sample contrast measured at the
same snapshot as the label. So the finding is an **association** ("pages that happen to be growing are longer"),
not evidence that length *causes* growth. One row per page, one 90-day window: nothing in this design can order
"longer -> later growth" in time.

**Does the validation carry the claim?** The samples are large but **unadjusted**. No controls for `content_type`,
`main_intent`, `avg_position`, or traffic. My own w04 audit showed missingness follows `content_type` — if the longer
*and* newer cohort is also the content type that gets keyword data, part of the word-count gap may be "which content
type the row is", not "length helps growth". **Question I'd ask the paper:** does the word-count gap (and the
freshness gap) survive controlling for content type, intent, and position tier — or is it a composition effect?
The code cell below reproduces the same means-table on our slice to show exactly what that table can and cannot say.

### Finding B — "The Freshness Multiplier" (paper Finding #4, CONFIRMED)

> **Paper:** "The most dramatic finding: 365+ day content that was refreshed within 30 days shows 3.2x health boost
> (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039). In this portfolio, refresh timing is one of the
> strongest measured levers available." — and in the same section: "the 361+ bucket is visible in the chart, but it is
> too small and too unstable to treat as a headline multiplier... spikes to 283:1 only because the sample is tiny and
> there is just 1 declining page in that bucket."

**Where does the label come from?** Two labels are at work. (1) "Refreshed within 30 days" is a proxy for
`days_since_last_update <= 30` — a *proxy* for an editorial act, because an editor choosing to refresh is not the same
as the page being reoptimized well. (2) "Health" is FlyRank's composite of impressions, position, and CTR
> **Paper (Optimization Flags):** "Internal FlyRank workflow labels... triage cues, not public definitions."
So the outcome is a **composite that already contains impressions** — and the headline claim is about *impressions*
(71 -> 4,039). Part of the "57x" is the outcome being inside the label.

**Does the validation carry the claim?** The 365+ refreshed-vs-untouched contrast is **not** a randomized refresh
experiment. Only pages someone already decided to rescue get refreshed — a selection effect. The paper itself flags the
bucket as too tiny (1 declining page in 361+), yet the "Expected:" line states the 3.2x as a lift an editor can count
on. **Question I'd ask the paper:** what does the same contrast look like when refreshed old pages are matched to
*comparable* untouched old pages (same age, same previous impressions, same position), and what is the base rate in
each bucket? My own staleness audit (w04) found freshness **MIXED** — only the severe 104+ day tail was elevated — so
this is exactly the comparison I wish the paper had shown.


In [1]:
# Section 1 evidence: reproduce the paper's cohort means-table on OUR slice, so the
# methodology question is concrete. Label = same family the paper uses (trend direction on impressions).
import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

df = pd.read_csv(Path(os.getcwd()) / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Loaded", len(df), "rows,", df["client_id"].nunique(), "clients; declining base rate", round(df["is_declining_label"].mean(), 3))
print("Trend direction counts:", df["trend_direction"].value_counts().to_dict())
print()

# The paper's Table (Finding #1): DIRECTION / COUNT / WORDS / AGE / AVG IMP / AVG POS / HEALTH
# We mirror the columns we actually have for the same growing-vs-declining split.
g = df[df["trend_direction"].isin(["up", "down"])].copy()
rows = {}
for d, ser in g.groupby("trend_direction"):
    rows[d] = {
        "count": len(ser),
        "avg word_count": ser["word_count"].mean(),
        "avg age_days": ser["content_age_days"].mean(),
        "avg impressions_90d": ser["impressions_90d"].mean(),
        "avg position": pd.to_numeric(ser["avg_position"], errors="coerce").replace(0, np.nan).mean(),
        "avg ctr": ser["ctr"].mean(),
    }
table = pd.DataFrame(rows).T.round(1)
print("Mirror of the paper's 'growing vs declining' cohort table (our slice, same label family):")
print(table.to_string())
print()
# The point: the cohort means differ --- but in WHICH direction is unstable across datasets.
wc_gap = (rows["up"]["avg word_count"] / rows["down"]["avg word_count"] - 1) * 100
age_gap = (rows["up"]["avg age_days"] / rows["down"]["avg age_days"] - 1) * 100
print(f"Our slice: growing pages are {wc_gap:.1f}% {'longer' if wc_gap >= 0 else 'shorter'} and "
      f"{age_gap:.1f}% {'older' if age_gap >= 0 else 'younger'} than declining pages.")
print("Direction note: the paper reported growing pages LONGER and YOUNGER; our slice shows the opposite on both.")
print("Same label family, opposite cohort composition --- exactly why an unadjusted means-table cannot carry")
print("a causal 'lengthen -> grows' claim, and why the paper's own 'observational comparison' caveat matters.")


Loaded 30000 rows, 32 clients; declining base rate 0.542
Trend direction counts: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}

Mirror of the paper's 'growing vs declining' cohort table (our slice, same label family):
        count  avg word_count  avg age_days  avg impressions_90d  avg position  avg ctr
down  16262.0          3221.8         236.2               4919.1          15.9      0.3
up     4388.0          2998.1         288.5               4716.1          22.5      0.6

Our slice: growing pages are -6.9% shorter and 22.1% older than declining pages.
Direction note: the paper reported growing pages LONGER and YOUNGER; our slice shows the opposite on both.
Same label family, opposite cohort composition --- exactly why an unadjusted means-table cannot carry
a causal 'lengthen -> grows' claim, and why the paper's own 'observational comparison' caveat matters.


## 2. My model under an honest split (before / after)

*Re-run your Week-5 model under the honest grouped-by-client split. Show both numbers.*

Week-5 used **GroupKFold(5) over train clients + a final held-out set of 8 of 32 clients** — pages of a client never
leak into the training fold that predicts them. The `before` number below is the naive **random row split** (25% of
*rows*, seed-fixed): pages from the same client sit on both sides, so the model can memorize *"which client owns this
page"* instead of *why* pages decline. The `after` is the grouped split. The **gap is the finding** — it is how much
accuracy the random split was borrowing from client identity.


In [2]:
# Section 2: the SAME Random Forest, SAME features, SAME metric; two split designs.
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 2026

# --- exact w05 feature vector (final, leakage-guarded) ---
feat = df.copy()
feat["has_keyword"] = feat["search_volume"].notna().astype(int)
feat["has_word_count"] = feat["word_count"].notna().astype(int)
feat["has_position"] = (feat["avg_position"] > 0).astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    feat[f"log_{c}"] = np.log1p(feat[c])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
HAS_FLAGS = ["has_keyword", "has_word_count", "has_position"]
for c in NUMERIC_FEATURES:
    feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for c in CATEGORICAL_FEATURES:
    feat[c] = feat[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

X = feat[NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS].copy()
y = feat["is_declining_label"]

def encode(Xdf):
    return pd.get_dummies(Xdf, columns=CATEGORICAL_FEATURES, drop_first=True)

def rf():
    return RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)

def prec_at_k(ranked, k):
    return ranked["y"].head(k).mean()

def evals(proba, test_y):
    rk = pd.DataFrame({"y": test_y.values, "p": proba}).sort_values("p", ascending=False)
    return pd.Series({"P@10": prec_at_k(rk, 10), "P@50": prec_at_k(rk, 50),
                      "AUC": roc_auc_score(test_y, proba), "test_rate": test_y.mean()})

# ---------- BEFORE: random row split ----------
Xe_all = encode(X)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(Xe_all, y, test_size=0.25, random_state=SEED)
m = rf(); m.fit(Xtr_r, ytr_r)
before = evals(m.predict_proba(Xte_r)[:, 1], yte_r)

# ---------- AFTER: grouped-by-client holdout (8 of 32 clients, seeded) ----------
rng = np.random.default_rng(SEED)
client_ids = pd.Series(df["client_id"].unique())
holdout = pd.Series(rng.choice(client_ids, size=int(len(client_ids) * 0.25), replace=False))
train_mask = ~df["client_id"].isin(holdout)
test_mask = df["client_id"].isin(holdout)
X_e_tr = encode(X[train_mask])
X_e_te = encode(X[test_mask]).reindex(columns=X_e_tr.columns, fill_value=0)
m2 = rf(); m2.fit(X_e_tr, y[train_mask])
after = evals(m2.predict_proba(X_e_te)[:, 1], y[test_mask])

print(f"Random-split rows: {len(Xtr_r)+len(Xte_r):,} -> train {len(Xtr_r):,} / test {len(Xte_r):,}  (BEFORE)")
print(f"Grouped-split: {int(train_mask.sum()):,} train / {int(test_mask.sum()):,} test pages, {len(holdout)} held-out clients (AFTER)")
comparison = pd.DataFrame({"Before (random split)": before, "After (grouped split)": after}).T.round(3)
print(comparison.to_string())
print()
print("The before/after of the improvement (read P@50 next to each test base rate):")
print(f"  AUC:  before {before['AUC']:.3f} (random rows) -> after {after['AUC']:.3f} (never-seen clients)")
print(f"  P@50: before {before['P@50']:.3f} on a {before['test_rate']:.3f} base -> "
      f"after {after['P@50']:.3f} on a {after['test_rate']:.3f} base")
print("  Reading: the grouped number keeps P@50 up even though its test pool has a HIGHER declining rate,")
print("  and the random split's extra +0.025 AUC was borrowed from client identity (memorization).")


Random-split rows: 30,000 -> train 22,500 / test 7,500  (BEFORE)
Grouped-split: 20,883 train / 9,117 test pages, 8 held-out clients (AFTER)
                       P@10  P@50    AUC  test_rate
Before (random split)   1.0  0.86  0.747      0.545
After (grouped split)   0.8  0.88  0.722      0.634

The before/after of the improvement (read P@50 next to each test base rate):
  AUC:  before 0.747 (random rows) -> after 0.722 (never-seen clients)
  P@50: before 0.860 on a 0.545 base -> after 0.880 on a 0.634 base
  Reading: the grouped number keeps P@50 up even though its test pool has a HIGHER declining rate,
  and the random split's extra +0.025 AUC was borrowed from client identity (memorization).


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Week-3 proved the confession: train the same classifier **once without** the suspect column (honest AUC ≈ 0.7) and
**once with** it (`trend_pct` — the label's own ingredient) and the AUC collapses toward 1.0. That, not the flattering
score, is the proof the guard is real. Below I re-run that hunt on the **final w05 vector** (same grouped split) plus
the two other families from the taxonomy: decision-derived columns (product flags / my w04 baseline score) and IDs.
Also a window check: the label is a 30d-vs-prev-30d trend inside the same trailing-90-day window as the features, so
every claim is a **concurrent association**, not a forward prediction.


In [3]:
# Section 3: leakage hunt on the FINAL feature set (same grouped split as Section 2).
LEAKY = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}

# 1) label-derived: honest vector vs vector + trend_pct
honest = evals(m2.predict_proba(X_e_te)[:, 1], y[test_mask])
leak_df = feat.copy()
leak_df["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
Xl_tr = pd.get_dummies(leak_df.loc[df.index[train_mask], NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS + ["trend_pct"]], columns=CATEGORICAL_FEATURES, drop_first=True)
Xl_te = pd.get_dummies(leak_df.loc[df.index[test_mask], NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS + ["trend_pct"]], columns=CATEGORICAL_FEATURES, drop_first=True).reindex(columns=Xl_tr.columns, fill_value=0)
ml = rf(); ml.fit(Xl_tr, y[train_mask])
leaky = evals(ml.predict_proba(Xl_te)[:, 1], y[test_mask])
print("Label-derived test (grouped split, same RF):")
print(f"  Honest vector            : AUC {honest['AUC']:.3f}")
print(f"  WITH trend_pct added     : AUC {leaky['AUC']:.3f}   <-- leak confession")
print("  VERDICT: AUC jumps toward 1.0 when the label's ingredient is added -> the guard works; trend family stays excluded.")
print()

# 2) decision-derived + IDs
in_vec = set(X.columns) | {"trend_pct" if False else "none"}
present = set(X.columns)
print("Decision-derived / ID / label-derived columns in the final vector:",
      sorted(present & (LEAKY | {"trend_pct"})) or "NONE")
print(f"  content_id in vector: {'content_id' in present} | client_id in vector: {'client_id' in present}")
print("  (client_id used only for the grouped split, never as a feature.)")
print()

# 3) window / timeline check
print("Window check:")
print("  Label: trend_direction = 30d-vs-prev-30d impression change, inside a single trailing-90d snapshot.")
print("  Features: all trailing-90d aggregates or static properties, knowable at export time.")
print("  VERDICT: no future window; but label and features share the same 90-day window -> associations are",
      "concurrent (observed, directional), NOT predictions of a future state.")


Label-derived test (grouped split, same RF):
  Honest vector            : AUC 0.722
  WITH trend_pct added     : AUC 1.000   <-- leak confession
  VERDICT: AUC jumps toward 1.0 when the label's ingredient is added -> the guard works; trend family stays excluded.

Decision-derived / ID / label-derived columns in the final vector: NONE
  content_id in vector: False | client_id in vector: False
  (client_id used only for the grouped split, never as a feature.)

Window check:
  Label: trend_direction = 30d-vs-prev-30d impression change, inside a single trailing-90d snapshot.
  Features: all trailing-90d aggregates or static properties, knowable at export time.
  VERDICT: no future window; but label and features share the same 90-day window -> associations are concurrent (observed, directional), NOT predictions of a future state.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence was in the Week-1 research-question notebook:

> **Original (w01):** "On the 30k-row anonymized slice, the random forest lifts Precision@50 from ~0.24 (baseline) to
> ~0.74 — a 3× improvement... catching ~37/50 means ~25 more correct priorities per 50 reviewed — directly translating
> to editor efficiency."

**Why it overreaches.** "3× improvement" and "catching 37/50" were written from an early exploratory run before the
honest validation existed; "directly translating to editor efficiency" asserts an operational payoff the model never
measured. **Safe rewrite:**

> **Rewritten:** "In an early exploratory (row-level) comparison the forest scored Precision@50 ≈ 0.74 vs ≈ 0.24 for an
> early rule variant. Under the honest grouped-by-client hold-out (w05), the observed figures were RF 0.88 vs baseline
> 0.92 on P@50 — the learned model **did not** beat the rule, so the ~3× figure was an artifact of a less careful
> comparison, not a held-out result. This is directional, decision-support evidence (the learned ranking sits next to
> the rule), and author-effort savings were not measured."

**The cause/evidence line:** a 3× number measured on a naive split *caused* an optimistic claim; the honest grouped
number is the *evidence* that keeps the claim at "observed, measured, directional, decision-support". The code cell
recomputes the held-out P@50 for both the rule and the forest so the rewritten numbers are produced in this run,
not copied.


In [4]:
# Section 4: recompute the numbers inside the rewrite so the safe claim is measured here, not copied.

def pct_rank(col):
    return pd.to_numeric(col, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def baseline_score(sub):
    on12 = (sub["avg_position"] > 0) & (sub["avg_position"] <= 20)
    return 0.85 * on12 * (1 - pct_rank(sub["ctr"])) + 0.10 * (on12 & (sub["days_since_last_update"] >= 104)).astype(int) \
        + 0.15 * pct_rank(np.log1p(sub["impressions_90d"]))

test_sub = df[test_mask].reset_index(drop=True).copy()
test_sub["baseline"] = baseline_score(test_sub)
rk = pd.DataFrame({"y": y[test_mask].values, "p": m2.predict_proba(X_e_te)[:, 1]}).sort_values("p", ascending=False)
b_ranked = test_sub.sort_values(["baseline", "impressions_90d"], ascending=[False, False])
print("On the 8 held-out clients (same pages, same metric, computed in this run):")
print(f"  Random Forest   P@50 = {prec_at_k(rk, 50):.3f}")
print(f"  Baseline rule   P@50 = {prec_at_k(pd.DataFrame({'y': b_ranked['is_declining_label'].values}), 50):.3f}")
print(f"  Test base rate        = {y[test_mask].mean():.3f}")
print()
print("Rewrite table:")
rewrite = pd.DataFrame({
    "claim": ["w01: 'RF lifts P@50 from ~0.24 to ~0.74, a 3x improvement'",
              "w01: 'catching ~37/50 -> ~25 more correct priorities'",
              "w05: 'decision-support while we hunt richer features'"],
    "what went too far": ["compared on a naive row-level run, not the honest split",
                          "declares an operational payoff the model never measured",
                          "fine claim; kept for contrast -- no learning score over-claims"],
    "safe rewrite": ["On the honest grouped-by-client hold-out: RF 0.88 vs rule 0.92 P@50 (observed).",
                     "P@50 is a ranking precision, not a count of saved editorial hours (not measured).",
                     "The learned ranking sits next to the rule as direction (observed, measured)."],
})
rewrite = rewrite.assign(kept_acc = ["no", "no", "yes"])
print(rewrite.to_string(index=False))
print()
print("Cause/evidence: the naive-split 3x number CAUSED the optimistic w01 claim; the grouped-split P@50 is the")
print("EVIDENCE that keeps my headline at 'observed, measured, directional, decision-support'.")


On the 8 held-out clients (same pages, same metric, computed in this run):
  Random Forest   P@50 = 0.880
  Baseline rule   P@50 = 0.920
  Test base rate        = 0.634

Rewrite table:
                                                     claim                                              what went too far                                                                      safe rewrite kept_acc
w01: 'RF lifts P@50 from ~0.24 to ~0.74, a 3x improvement'        compared on a naive row-level run, not the honest split   On the honest grouped-by-client hold-out: RF 0.88 vs rule 0.92 P@50 (observed).       no
     w01: 'catching ~37/50 -> ~25 more correct priorities'        declares an operational payoff the model never measured P@50 is a ranking precision, not a count of saved editorial hours (not measured).       no
     w05: 'decision-support while we hunt richer features' fine claim; kept for contrast -- no learning score over-claims      The learned ranking sits next to the rule as dire

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two paper findings named (Anatomy of Growing Content; Freshness Multiplier), each with a label-provenance and a validation question
- [x] My model re-run under the honest grouped-by-client split with a before (random) / after (grouped) table — the gap is the finding
- [x] Leakage audit re-run on the final feature set: label-derived confession (AUC -> ~1.0 when trend_pct added), no decision-derived or ID columns, window/timeline verdict
- [x] Boldest claim rewritten in safe language with the cause/evidence distinction made explicit
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


In [5]:
# validation marker — this cell runs last and stays empty by design
